# ☁️ Chapter 9: Cloud Computing

📖 Book: Chapter 9, pp. 344–391 · ⏱️ 2–3 hours · 🧱 Needs: Ch 3, 7 · Level: Intermediate

---

> **🎯 Key idea:** Choose `ThreadPoolExecutor` for I/O-bound cloud APIs, `multiprocessing` for CPU work, and `boto3` paginators/retries/IAM roles for reliability.

> **💡 Tip:** Run cells sequentially top-to-bottom (`Shift + Enter`).

> **⚠️ Warning:** Cells tagged `external` may require extra packages; cloud stacks incur costs if deployed.

## 🔄 Then vs. Now

> **📜 Then (book, ~2020):** Raw `list_objects_v2()` + static keys  
> **🆕 Now (2026):** `get_paginator()`, `botocore.config.Config`, IAM roles  
> **Why:** Avoids 1,000-item truncation & leaked keys

> **📜 Then (book, ~2020):** Live AWS account only  
> **🆕 Now (2026):** `moto` mock library + unit tests  
> **Why:** Zero cost & fast CI feedback

---

## 4. 💡 Conceptual Cards

### 🚨 Real-World Problem
You need to check the health of 20 cloud regions or list 50,000 S3 objects, and your sequential script takes minutes or silently stops at 1,000 items. Meanwhile, a transient AWS `ThrottlingException` crashes the whole job. This chapter teaches when to use threads vs. processes, exponential backoff, and `boto3` paginators.

### Concept: GIL & Concurrency Choice
**What:** `ThreadPoolExecutor` vs. `multiprocessing.Pool`  
**Why:** GIL blocks CPU threads, not network I/O waits  
**How:** Threads share memory for I/O; Pools spawn processes  
**When:** Threads for HTTP/`boto3`; Pool for CPU

📖 *Book listing:* `multiprocessing.Pool` KMeans Clustering (pp. 378–386)

**Plain-language analogy:** The GIL means one chef (`threading`) can only chop one vegetable at a time, but while waiting for water to boil (network I/O), that chef can check five other pots (`ThreadPoolExecutor`).

**Walkthrough:** The book bypasses CPython's GIL for CPU-heavy math by fanning work across processes with `multiprocessing.Pool.map()`, while noting `ThreadPoolExecutor` fits network I/O.

### Concept: Exponential Backoff & Retries
**What:** Wait `2^n` + jitter on `ThrottlingException`  
**Why:** Prevents retry storms against cloud APIs  
**How:** `Config(retries={'mode': 'adaptive'})`  
**When:** Every production cloud client

📖 *Book listing:* AWS Lambda Wikipedia Handler & `subprocess` Timeout (pp. 358–377)

**Plain-language analogy:** Exponential backoff is like politely waiting 1s, then 2s, then 4s before knocking again on a busy office door instead of pounding nonstop.

**Walkthrough:** The book wraps external commands with `timeout=` to catch `TimeoutExpired`, and structures stateless Lambda handlers returning JSON-serialized `statusCode` and `body`.

## 6. 🟢 Beginner Examples

#### Example 1: Beginner Example 1: Safe subprocess with timeout
*Expected output:* Printed labelled output below.

In [1]:
# Beginner Example 1: Safe subprocess with timeout
import logging, subprocess

try:
    cp = subprocess.run(['python3', '--version'], capture_output=True, text=True, timeout=5, check=True)
    print('Subprocess:', cp.stdout.strip())
except subprocess.TimeoutExpired:
    logging.exception('Timed out')

Subprocess: Python 3.13.14


**What the output means:** Shows the exact exception caught cleanly instead of crashing the whole script.

> **🧪 Try it:** Change the invalid input to a valid one and observe the `except` branch no longer triggers.

#### Example 2: Beginner Example 2: CPU-bound multiprocessing.Pool
*Expected output:* Printed labelled output below.

In [2]:
# Beginner Example 2: CPU-bound multiprocessing.Pool
import math
from multiprocessing import Pool

if __name__ == '__main__':
    with Pool(2) as pool:
        print('Pool sqrt:', pool.map(math.sqrt, [4, 9, 16, 25]))

Pool sqrt: [2.0, 3.0, 4.0, 5.0]


**What the output means:** The labelled `print(...)` lines confirm the transformed data or status returned by the logic above.

> **🧪 Try it:** Modify one input value above and re-run (`Shift + Enter`) to verify how the output changes.

#### Example 3: Beginner Example 3: I/O-bound ThreadPoolExecutor across regions
*Expected output:* Printed labelled output below.

In [3]:
# Beginner Example 3: I/O-bound ThreadPoolExecutor across regions
import time
from concurrent.futures import ThreadPoolExecutor

def check_region(region: str) -> str:
    time.sleep(0.01)
    return f'{region}: healthy'

with ThreadPoolExecutor(max_workers=3) as ex:
    print('Regions:', list(ex.map(check_region, ['us-east-1', 'us-west-2', 'eu-west-1'])))

Regions: ['us-east-1: healthy', 'us-west-2: healthy', 'eu-west-1: healthy']


**What the output means:** The labelled `print(...)` lines confirm the transformed data or status returned by the logic above.

> **🧪 Try it:** Modify one input value above and re-run (`Shift + Enter`) to verify how the output changes.

## 7. 🟡 Intermediate Examples: Exponential Backoff & Modern `boto3` (`Config` + Paginator + `moto`)

#### Example 4: Intermediate Example 1: Exponential backoff retry for transient cloud throttling
*Expected output:* Printed labelled output below.

In [4]:
# Intermediate Example 1: Exponential backoff retry for transient cloud throttling
import time

def retry_backoff(fn, retries: int = 3, base_delay: float = 0.01):
    for attempt in range(1, retries + 1):
        try:
            return fn()
        except RuntimeError:
            if attempt == retries:
                raise
            time.sleep(base_delay * (2 ** (attempt - 1)))

state = {'calls': 0}
def flaky_api():
    state['calls'] += 1
    if state['calls'] < 3:
        raise RuntimeError('ThrottlingException')
    return '200 OK'

print('Retry result:', retry_backoff(flaky_api), '| attempts:', state['calls'])

Retry result: 200 OK | attempts: 3


**What the output means:** Shows the exact exception caught cleanly instead of crashing the whole script.

> **🧪 Try it:** Change the invalid input to a valid one and observe the `except` branch no longer triggers.

> **⚠️ Warning:** Cloud commands incur costs—clean up test resources afterward.

#### Example 5: Intermediate Example 2 (External + Mock Fallback): boto3 Config, Paginator & moto
*Expected output:* Simulated fallback if optional package is missing.

In [5]:
# Intermediate Example 2 (External + Mock Fallback): boto3 Config, Paginator & moto
# Requires: pip install boto3 moto
# WARNING: Never run against a real AWS account without checking cost & cleaning up buckets/instances!
try:
    import boto3
    from botocore.config import Config
    cfg = Config(retries={'max_attempts': 5, 'mode': 'adaptive'})
    print('boto3 Config ready:', cfg.retries)
except ImportError:
    mock_pages = [{'Contents': [{'Key': 'logs/2026.gz'}]}, {'Contents': [{'Key': 'logs/2027.gz'}]}]
    keys = [obj['Key'] for page in mock_pages for obj in page.get('Contents', [])]
    print('boto3 not installed; simulated paginated S3 keys:', keys)

boto3 not installed; simulated paginated S3 keys: ['logs/2026.gz', 'logs/2027.gz']


**What the output means:** If the optional library is installed, it runs live; otherwise the `except ImportError:` fallback prints a safe offline simulation.

> **🧪 Try it:** Install the optional package in your `.venv` (see the comment at top of the cell) and re-run to compare live vs. simulated output.

## 8. 🚀 Real-World DevOps Applications (Personal DevOps Learning Track)
*Supplementary extension:* **Spot vs. On-Demand Cost Estimator** for stateless batch workloads.

#### Example 6: def estimate_spot_savings(hourly_on_demand: float, hours: int, discount: float = 0.7) -> dict:
*Expected output:* Printed labelled output below.

In [6]:
def estimate_spot_savings(hourly_on_demand: float, hours: int, discount: float = 0.7) -> dict:
    od = round(hourly_on_demand * hours, 2)
    spot = round(od * (1 - discount), 2)
    return {'on_demand': od, 'spot': spot, 'saved': round(od - spot, 2)}

print('730h savings:', estimate_spot_savings(0.192, 730))

730h savings: {'on_demand': 140.16, 'spot': 42.05, 'saved': 98.11}


**What the output means:** The labelled `print(...)` lines confirm the transformed data or status returned by the logic above.

> **🧪 Try it:** Modify one input value above and re-run (`Shift + Enter`) to verify how the output changes.

#### Real Tool Demo (guarded with `shutil.which`)
*Expected output:* CLI version or check output (or graceful skip).

In [7]:
# Real Tool Demo: aws CLI check (guarded with shutil.which)
import shutil, subprocess
if shutil.which('aws'):
    print('AWS CLI:', subprocess.run(['aws', '--version'], capture_output=True, text=True).stdout.strip())
else:
    print('aws CLI not installed; using moto/mock fallbacks')

AWS CLI: aws-cli/2.35.24 Python/3.14.6 Linux/7.0.0-44-generic exe/x86_64.ubuntu.26


**What the output means:** Checks whether the real CLI tool exists on `PATH`; if found, runs a safe read-only/dry-run check, otherwise skips cleanly.

> **🧪 Try it:** Try adding `--help` or inspecting `cp.returncode` to see how exit codes behave.

---

## 9. ⚠️ Common Mistakes & Troubleshooting

- ❌ **Hardcoding AWS keys** -> Credential compromise in Git. ✅ Use IAM roles / OIDC / env vars.
- ❌ **Omitting `get_paginator()`** -> Silently misses keys > 1,000. ✅ Always paginate list APIs.
- ❌ **Lambdas in `Pool.map` (3.14)** -> `PicklingError` under `forkserver`. ✅ Use top-level functions or built-ins.

> **🧪 Try it:** Run the starter cell below—it prints `TODO` until you implement the body.

#### 🧪 Starter Exercise: `Starter Exercise Cell (implement classify_cloud_service and run to verify!)`
*Expected output:* `TODO` message until implemented, then `✅ Starter checks PASSED!`.

In [8]:
# Starter Exercise Cell (implement classify_cloud_service and run to verify!)
def _run_starter_checks():
    assert classify_cloud_service('Lambda') == 'FaaS'
def classify_cloud_service(name: str) -> str:
    raise NotImplementedError('Implement classify_cloud_service')

try:
    _run_starter_checks()
    print('✅ Starter checks PASSED!')
except NotImplementedError as e:
    print(f'🧪 TODO: {e}')

🧪 TODO: Implement classify_cloud_service


## 12. 🧠 Knowledge Checks (Inline Q&A for Practice)
**Q1:** Why doesn't Python `threading` speed up CPU-bound math loops?

<details>
<summary>✅ Answer 1</summary>

The CPython GIL allows only one thread to execute Python bytecode at a time.

</details>

**Q2:** When IS `ThreadPoolExecutor` ideal?

<details>
<summary>✅ Answer 2</summary>

I/O-bound workloads (HTTP, `boto3`, DB queries) where threads wait on network sockets.

</details>

**Q3:** Why use `boto3` paginators?

<details>
<summary>✅ Answer 3</summary>

Single `list_objects_v2` calls cap at 1,000 items; paginators fetch all pages automatically.

</details>

**Q4:** What does Amdahl's Law state?

<details>
<summary>✅ Answer 4</summary>

Maximum speedup is capped by the non-parallelizable sequential portion of a workload.

</details>

**Q5:** Why are Spot instances bad for single-node stateful databases?

<details>
<summary>✅ Answer 5</summary>

Cloud providers can reclaim Spot capacity with ~2 minutes notice.

</details>

## 13. 🔑 Hints & Complete Solutions

<details>
<summary>💭 Hints (click to expand)</summary>

- **Challenge**: Write `evaluate_region(rec)` and pass it to `ThreadPoolExecutor.map`.

</details>

> **💡 Tip:** Complete solutions are in the collapsed/marked cell below.

<details>
<summary>✅ Complete Solutions (expand/run cell below to verify)</summary>

Run the verified solutions cell below to check all exercise & challenge assertions.

</details>

In [9]:
# Complete Solutions + Assert Verification
import random, subprocess, time
from concurrent.futures import ThreadPoolExecutor

def run_timed(cmd: list[str], timeout: int = 2) -> str:
    try:
        return subprocess.run(cmd, capture_output=True, text=True, timeout=timeout, check=True).stdout.strip()
    except subprocess.TimeoutExpired:
        return 'TIMEOUT'

def classify_cloud_service(name: str) -> str:
    return {'EC2': 'IaaS', 'S3': 'IaaS', 'Lambda': 'FaaS', 'App Engine': 'PaaS'}.get(name, 'Unknown')

def parallel_lengths(items: list[str]) -> list[int]:
    with ThreadPoolExecutor(max_workers=4) as ex:
        return list(ex.map(len, items))

def filter_running_micro(instances: list[dict]) -> list[str]:
    return [i['Id'] for i in instances if i.get('State') == 'running' and i.get('Type') == 't3.micro']

def evaluate_region(rec: tuple[str, int, int]) -> dict:
    region, latency, code_ = rec
    return {'region': region, 'ok': (code_ == 200 and latency <= 250), 'latency': latency}

def aggregate_health(records: list[tuple[str, int, int]]) -> dict:
    with ThreadPoolExecutor(max_workers=4) as ex:
        results = list(ex.map(evaluate_region, records))
    return {'status': 'HEALTHY' if all(r['ok'] for r in results) else 'DEGRADED', 'regions': results}

assert 'Python' in run_timed(['python3', '--version'])
assert classify_cloud_service('Lambda') == 'FaaS'
assert parallel_lengths(['a', 'abc']) == [1, 3]
assert filter_running_micro([{'Id': 'i-1', 'State': 'running', 'Type': 't3.micro'}]) == ['i-1']
assert aggregate_health([('us-east-1', 80, 200), ('eu-west-1', 310, 200)])['status'] == 'DEGRADED'
print('All Chapter 9 assertions PASSED!')

All Chapter 9 assertions PASSED!


---

## 14. 📌 Chapter Summary & Quick Reference

- `ThreadPoolExecutor(max_workers=4)` — Parallel cloud API I/O (e.g. `ex.map(check_region, regions)`)
- `s3.get_paginator('list_objects_v2')` — Fetch all S3 pages (e.g. `for page in paginator.paginate(...)`)
- `subprocess.run(..., timeout=5)` — Prevent hung CLI calls (e.g. Catch `TimeoutExpired`)

### ✅ What You Can Now Do
- Pick `ThreadPoolExecutor` for I/O-bound APIs and `multiprocessing.Pool` for CPU math.
- Implement exponential backoff retries for transient cloud throttling.
- Use modern `boto3` paginators, `Config` retries, IAM roles, and `moto` offline testing.

### ➡️ Next Chapter
Chapter 10 builds on this by replacing imperative cloud scripts with declarative Terraform/OpenTofu and Pulumi IaC.